# Chapter 10. The Mobile Channel and Doppler

The code from Chapter 10 of *Radio Propagation Ray Tracing with Sionna RT*.
The cells follow the order of the text, and the contents of each code cell are
the same as printed there.

Only the first Setup cell is not printed in the book. It performs the imports
and the scene loading that the printed code assumes. Once the setup cell has
run, the rest can be run from the top in order.

Fragments the text shows for explanation, such as a path to a scene of your own
or an object name that scene does not contain, will not run as they stand. A
note appears before each such cell.

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import sionna.rt as rt
from sionna.rt import (load_scene, scene as scene_assets,
                       Transmitter, Receiver, PlanarArray,
                       PathSolver, RadioMapSolver, Camera, RadioMaterial)

# the central Munich urban scene used in many chapters of this book
scene = load_scene(scene_assets.munich)
scene.frequency = 3.5e9
scene.tx_array = PlanarArray(num_rows=1, num_cols=1,
                             pattern='iso', polarization='V')
scene.rx_array = PlanarArray(num_rows=1, num_cols=1,
                             pattern='iso', polarization='V')
scene.add(Transmitter('tx0', position=[-30.0, 38.0, 35.0]))

solver = PathSolver()
rmsolver = RadioMapSolver()

# giving the receiver a velocity yields the Doppler shift per path
scene.add(Receiver('rx0', position=[60.0, -40.0, 1.5],
                   velocity=[10.0, 0.0, 0.0]))
paths = solver(scene, max_depth=5)
a, tau = paths.cir(normalize_delays=True, out_type='numpy')

## 2. Moving Objects

**Note**: object names differ by scene. Check the actual names with `scene.objects` and substitute.

In [ ]:
car = scene.objects['car-001']   # object names differ by scene
car.velocity = [10.0, 0.0, 0.0]  # 10 m/s in +x

## 3. Setting the Velocity Vector

In [ ]:
rx = Receiver('rx0', position=[100.0, 50.0, 1.5],
              velocity=[10.0, 0.0, 0.0])  # 10 m/s along x
scene.add(rx)

## 4. Doppler Shift per Path

In [ ]:
# Doppler per path, computed with velocity set on the Receiver
fD = np.array(paths.doppler).flatten()
valid = np.array(paths.valid).flatten()
fD_valid = fD[valid]
print('Doppler shift range:', fD_valid.min(), 'to',
      fD_valid.max(), 'Hz')

## 5. Building a Time-Varying CIR

In [ ]:
# generate a time-varying CIR over 0.1 s at 1 ms samples
t_axis = np.linspace(0, 0.1, 100)
H_t = []
for t in t_axis:
    # rotate each path's phase by v . k_hat * t
    a_t = a * np.exp(1j*2*np.pi*fD*t)
    H_t.append(a_t)
H_t = np.array(H_t)  # (num_time, num_paths)

## 7. Simulating Vehicle Motion

In [ ]:
v_kmh = 36; v = v_kmh/3.6  # m/s
trajectory = lambda t: [v*t - 200, 50, 1.5]
cir_seq = []
for t in np.linspace(0, 30, 151):  # 30 s at 0.2 s steps
    scene.remove('rx0')
    scene.add(Receiver('rx0', position=trajectory(t)))
    paths = solver(scene, max_depth=3)
    cir_seq.append(paths)